### PySpark Otomoto Demo 

Źródło danych: https://www.kaggle.com/datasets/szymoncyperski/car-sales-offers-from-otomotopl-2023 


In [ ]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

**Teoria:** Powyżej importujemy niezbędne biblioteki. `SparkSession` to główny punkt wejścia do funkcjonalności DataFrame i SQL w Sparku (od wersji 2.0). Moduł `functions` dostarcza wbudowane funkcje operujące na kolumnach, a `matplotlib.pyplot` posłuży nam do późniejszej wizualizacji danych.


In [ ]:
spark = SparkSession.builder \
    .appName("Otomoto Demo") \
    .getOrCreate()


**Teoria:** Tworzymy sesję Sparka. `builder` używa wzorca projektowego Builder do skonfigurowania sesji. `getOrCreate()` tworzy nową sesję lub pobiera istniejącą, co jest bezpieczne przy wielokrotnym uruchamianiu notatnika.


In [ ]:
df = spark.read.option("header", True) \
    .option("delimiter", ";") \
    .option("inferSchema", False) \
    .csv("otomoto_offers_eng_23-04-2023.csv")


**Teoria:** Wczytywanie danych. Spark używa leniwego ewaluowania (lazy evaluation) - dane nie są fizycznie wczytywane w tym momencie, tworzony jest tylko plan wykonania (DAG). Ustawiamy `header=True` ponieważ nasz plik CSV ma nagłówki, oraz określamy separator jako średnik `;`.


In [ ]:
df.show()

**Teoria:** `show()` to akcja (action), która uruchamia wykonanie obliczeń w Sparku. Dopiero teraz plik jest odczytywany, a wynik prezentowany na ekranie.


In [ ]:
df.filter(F.col("vehicle_brand") == "Volvo").show()

In [ ]:
df = df.withColumn("price_num",
                   F.regexp_replace(F.col("price"), r"[^\d]", "").cast("double"))

df = df.withColumn("mileage_km",
                   F.regexp_replace(F.col("mileage"), r"[^\d]", "").cast("integer"))

df = df.withColumn("production_year_int",
                   F.regexp_replace(F.col("production_year"), r"[^\d]", "").cast("integer"))

df = df.withColumn("engine_cc",
                   F.regexp_replace(F.col("engine_displacement"), r"[^\d]", "").cast("integer"))

df = df.withColumn("power_hp",
                   F.regexp_replace(F.col("power"), r"[^\d]", "").cast("integer"))

df = df.withColumn("fuel_clean",
                   F.lower(F.trim(F.col("fuel_type"))))

In [ ]:
df.select("vehicle_brand", "vehicle_model", "price_num", "mileage_km",
          "production_year_int", "engine_cc", "power_hp", "fuel_clean") \
  .show(10, truncate=False)

**Teoria:** `select()` to transformacja, która działa jak w SQL - pozwala wybrać podzbiór kolumn. Zmniejsza to ilość przetwarzanych danych w dalszych krokach.


In [ ]:
avg_brand = df.groupBy("vehicle_brand") \
              .agg(F.round(F.avg("price_num"), 2).alias("avg_price")) \
              .orderBy(F.col("avg_price").desc())

print("Średnia cena per marka")
avg_brand.show(20, truncate=False)

In [ ]:
fuel_count = df.groupBy("fuel_clean").count()
print("Liczba ogłoszeń wg rodzaju paliwa")
fuel_count.show()

In [ ]:
df.createOrReplaceTempView("cars")

In [ ]:
# SQL: zależność mocy i pojemności od ceny
spark.sql("""
    SELECT vehicle_brand,
           ROUND(AVG(power_hp), 1) AS avg_power,
           ROUND(AVG(engine_cc), 1) AS avg_cc,
           ROUND(AVG(price_num), 1) AS avg_price
    FROM cars
    GROUP BY vehicle_brand
    ORDER BY avg_power DESC
""").show()

In [ ]:
df.groupBy("production_year_int") \
  .count() \
  .orderBy(F.col("production_year_int").desc()) \
  .show()

In [ ]:
# Średnia cena i przebieg per marka i model
df.groupBy("vehicle_brand", "vehicle_model") \
  .agg(
      F.round(F.avg("price_num"), 2).alias("avg_price"),
      F.round(F.avg("mileage_km"), 2).alias("avg_mileage")
  ) \
  .orderBy(F.col("avg_price").desc()) \
  .show(20, truncate=False)

In [ ]:
# zależność ceny od przebiegu 
price_mileage = df.select("price_num", "mileage_km") \
                  .where((F.col("price_num").isNotNull()) & (F.col("mileage_km").isNotNull()))

In [ ]:
pdf_scatter = price_mileage.sample(fraction=0.1, seed=42).toPandas()

plt.figure(figsize=(8,5))
plt.scatter(pdf_scatter["mileage_km"], pdf_scatter["price_num"], s=6)
plt.title("Cena vs Przebieg")
plt.xlabel("Przebieg [km]")
plt.ylabel("Cena")
plt.tight_layout()
plt.savefig("scatter_price_mileage.png")

print("Wizualizacja scatter zapisana jako scatter_price_mileage.png")

**Teoria:** `toPandas()` to akcja, która zbiera (collect) wszystkie dane na partycjach roboczych i przesyła je na węzeł główny (Driver), konwertując do struktury Pandas DataFrame. Uwaga: Można tego używać tylko na małych zbiorach (po limitowaniu np. top 10), w przeciwnym razie braknie pamięci RAM na Driverze!


---
# Zadanie samodzielne: Analiza Przestępczości w Chicago

Poniżej znajduje się miejsce na realizację zadania z analizy danych przy użyciu PySpark na zbiorze *Chicago Crimes* (około 50 000 ostatnich zdarzeń). Twoim celem jest przygotowanie, wyczyszczenie oraz zanalizowanie tych danych z wykorzystaniem zaawansowanych optymalizacji dostępnych w Sparku.

### Wymagania:
1. **Wczytanie i Czyszczenie Danych:** Wczytaj pobrany plik `chicago_crimes_sample.csv`. Usuń ewentualne duplikaty, wiersze z brakami danych (szczególnie w kluczowych kolumnach) i odfiltruj/napraw błędne daty.
2. **UDF i Pora Dnia:** Dodaj nową kolumnę z klasyfikacją pory dnia (np. noc, dzień, wieczór) utworzoną za pomocą User Defined Function (UDF) w oparciu o godzinę z kolumny `Date`.
3. **Optymalizacja i Partycjonowanie:** Zoptymalizuj przetwarzanie. Zastanów się, w których momentach użyć `cache()`. Przy dołączaniu mniejszych tabel słownikowych (jeśli byś je tworzył), wykorzystaj *broadcast join*. Ostatecznie zapisz przefiltrowane dane do formatu **Parquet** z podziałem na partycje według roku (`Year`).
4. **Analiza i Plany Zapytań:** Przeprowadź analizę statystyczną przestępstw (np. jakiego typu przestępstwa są najpopularniejsze w konkretnych lokacjach, o konkretnym czasie). Wykorzystaj funkcję `.explain()` aby udokumentować plan zapytania Sparka dla najcięższej agregacji.
5. *(Opcjonalnie)* **Uczenie Maszynowe (MLlib):** Spróbuj zbudować i wytrenować prosty model wieloklasowy, przewidujący rodzaj przestępstwa (`Primary Type`) na podstawie innych atrybutów, jak lokacja, godzina, arrest itp.

## Rozwiązanie zadania

Poniżej jest jedna spójna część rozwiązania, podzielona komentarzami na podpunkty z treści zadania. Kod jest celowo prosty i podobny stylem do wcześniejszych przykładów z laboratorium: DataFrame, `groupBy`, `agg`, `show`, `cache`, `broadcast` i zapis do Parquet.

In [ ]:
import os
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StringType

# Konfiguracja Sparka dla lokalnego uruchamiania w notebooku.
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("ChicagoCrimes") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# 1. Wczytanie i czyszczenie danych
csv_path = Path("chicago_crimes_sample.csv")
if not csv_path.exists():
    csv_path = Path("materialy/lab_05/chicago_crimes_sample.csv")

df_raw = spark.read \
    .option("header", True) \
    .option("multiLine", True) \
    .option("escape", '"') \
    .csv(str(csv_path))

print("Surowe dane:")
df_raw.show(5, truncate=False)

df_clean = df_raw.select(
        F.col("id").cast("long").alias("id"),
        F.trim(F.col("case_number")).alias("case_number"),
        F.to_timestamp("date", "yyyy-MM-dd'T'HH:mm:ss.SSS").alias("date"),
        F.trim(F.col("block")).alias("block"),
        F.trim(F.col("primary_type")).alias("primary_type"),
        F.trim(F.col("description")).alias("description"),
        F.trim(F.col("location_description")).alias("location_description"),
        F.col("arrest").cast("boolean").alias("arrest"),
        F.col("domestic").cast("boolean").alias("domestic"),
        F.trim(F.col("district")).alias("district"),
        F.col("ward").cast("int").alias("ward"),
        F.col("latitude").cast("double").alias("latitude"),
        F.col("longitude").cast("double").alias("longitude")
    ) \
    .withColumn("year", F.year("date")) \
    .dropDuplicates(["id", "case_number"]) \
    .dropna(subset=["id", "case_number", "date", "primary_type", "location_description", "year"]) \
    .where((F.col("year") >= 2001) & (F.col("year") <= F.year(F.current_date()))) \
    .where(
        (F.col("latitude").isNull() | F.col("latitude").between(-90, 90)) &
        (F.col("longitude").isNull() | F.col("longitude").between(-180, 180))
    )

print("Liczba wierszy przed czyszczeniem:", df_raw.count())
print("Liczba wierszy po czyszczeniu:", df_clean.count())
df_clean.show(5, truncate=False)


# 2. UDF i pora dnia
def pora_dnia(hour):
    if hour is None:
        return None
    if 5 <= hour < 12:
        return "rano"
    if 12 <= hour < 17:
        return "dzień"
    if 17 <= hour < 22:
        return "wieczór"
    return "noc"

pora_dnia_udf = F.udf(pora_dnia, StringType())

df_time = df_clean \
    .withColumn("hour", F.hour("date")) \
    .withColumn("pora_dnia", pora_dnia_udf(F.col("hour")))

print("Liczba zdarzeń według pory dnia:")
df_time.groupBy("pora_dnia").count().orderBy(F.desc("count")).show()


# 3. Optymalizacja i partycjonowanie
crime_groups = spark.createDataFrame(
    [
        ("THEFT", "mienie"),
        ("BURGLARY", "mienie"),
        ("ROBBERY", "mienie"),
        ("MOTOR VEHICLE THEFT", "mienie"),
        ("CRIMINAL DAMAGE", "mienie"),
        ("ASSAULT", "przemoc"),
        ("BATTERY", "przemoc"),
        ("HOMICIDE", "przemoc"),
        ("NARCOTICS", "narkotyki"),
        ("DECEPTIVE PRACTICE", "oszustwa"),
        ("WEAPONS VIOLATION", "broń"),
    ],
    ["primary_type", "crime_group"]
)

df_ready = df_time \
    .join(F.broadcast(crime_groups), on="primary_type", how="left") \
    .fillna({"crime_group": "inne"}) \
    .cache()

print("Liczba wierszy gotowych do analizy:", df_ready.count())

parquet_path = csv_path.parent / "chicago_crimes_parquet"
df_ready.write \
    .mode("overwrite") \
    .partitionBy("year") \
    .parquet(str(parquet_path))

print("Zapisano dane Parquet do:", parquet_path)


# 4. Analiza statystyczna i plan zapytania
print("Najczęstsze typy przestępstw:")
df_ready.groupBy("primary_type") \
    .agg(F.count("*").alias("liczba")) \
    .orderBy(F.desc("liczba")) \
    .show(10, truncate=False)

print("Najczęstsze lokalizacje:")
df_ready.groupBy("location_description") \
    .agg(F.count("*").alias("liczba")) \
    .orderBy(F.desc("liczba")) \
    .show(10, truncate=False)

print("Typy przestępstw według pory dnia:")
df_ready.groupBy("pora_dnia", "primary_type") \
    .agg(F.count("*").alias("liczba")) \
    .orderBy("pora_dnia", F.desc("liczba")) \
    .show(30, truncate=False)

heavy_aggregation = df_ready.groupBy("location_description", "pora_dnia", "primary_type") \
    .agg(
        F.count("*").alias("liczba"),
        F.round(F.avg(F.col("arrest").cast("double")), 3).alias("arrest_rate")
    ) \
    .orderBy(F.desc("liczba"))

print("Plan zapytania dla najcięższej agregacji:")
heavy_aggregation.explain(mode="formatted")

print("Wynik najcięższej agregacji:")
heavy_aggregation.show(20, truncate=False)

window = Window.partitionBy("location_description", "pora_dnia").orderBy(F.desc("liczba"))

top_by_location_time = heavy_aggregation \
    .withColumn("rank", F.row_number().over(window)) \
    .where(F.col("rank") <= 3) \
    .orderBy("location_description", "pora_dnia", "rank")

print("Top 3 typy przestępstw według lokalizacji i pory dnia:")
top_by_location_time.show(50, truncate=False)

Surowe dane:
+--------+-----------+-----------------------+---------------------+----+------------+---------------------------------------+---------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-----------------------+------------+-------------+------------------------------------+
|id      |case_number|date                   |block                |iucr|primary_type|description                            |location_description |arrest|domestic|beat|district|ward|community_area|fbi_code|x_coordinate|y_coordinate|year|updated_on             |latitude    |longitude    |location                            |
+--------+-----------+-----------------------+---------------------+----+------------+---------------------------------------+---------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-----------------------+------------+-------------+------------------------------------+
|14193

Liczba wierszy gotowych do analizy: 49803


Zapisano dane Parquet do: chicago_crimes_parquet
Najczęstsze typy przestępstw:
+-------------------+------+
|primary_type       |liczba|
+-------------------+------+
|THEFT              |10920 |
|BATTERY            |9253  |
|CRIMINAL DAMAGE    |5497  |
|ASSAULT            |4564  |
|MOTOR VEHICLE THEFT|3886  |
|OTHER OFFENSE      |3459  |
|BURGLARY           |3163  |
|DECEPTIVE PRACTICE |2486  |
|NARCOTICS          |1455  |
|CRIMINAL TRESPASS  |1193  |
+-------------------+------+
only showing top 10 rows
Najczęstsze lokalizacje:
+--------------------------------------+------+
|location_description                  |liczba|
+--------------------------------------+------+
|STREET                                |13979 |
|APARTMENT                             |9802  |
|RESIDENCE                             |5692  |
|SIDEWALK                              |2308  |
|PARKING LOT / GARAGE (NON RESIDENTIAL)|1842  |
|SMALL RETAIL STORE                    |1708  |
|DEPARTMENT STORE                

+---------+---------------------------------+------+
|pora_dnia|primary_type                     |liczba|
+---------+---------------------------------+------+
|dzień    |THEFT                            |3640  |
|dzień    |BATTERY                          |2294  |
|dzień    |ASSAULT                          |1377  |
|dzień    |CRIMINAL DAMAGE                  |1042  |
|dzień    |OTHER OFFENSE                    |957   |
|dzień    |DECEPTIVE PRACTICE               |948   |
|dzień    |MOTOR VEHICLE THEFT              |682   |
|dzień    |BURGLARY                         |541   |
|dzień    |NARCOTICS                        |510   |
|dzień    |CRIMINAL TRESPASS                |357   |
|dzień    |WEAPONS VIOLATION                |230   |
|dzień    |ROBBERY                          |199   |
|dzień    |OFFENSE INVOLVING CHILDREN       |128   |
|dzień    |SEX OFFENSE                      |87    |
|dzień    |PUBLIC PEACE VIOLATION           |78    |
|dzień    |INTERFERENCE WITH PUBLIC OFFICER |6

+--------------------+---------+-------------------+------+-----------+
|location_description|pora_dnia|primary_type       |liczba|arrest_rate|
+--------------------+---------+-------------------+------+-----------+
|STREET              |noc      |MOTOR VEHICLE THEFT|992   |0.034      |
|APARTMENT           |noc      |BATTERY            |971   |0.249      |
|STREET              |wieczór  |MOTOR VEHICLE THEFT|922   |0.026      |
|STREET              |noc      |CRIMINAL DAMAGE    |795   |0.045      |
|APARTMENT           |wieczór  |BATTERY            |773   |0.176      |
|APARTMENT           |rano     |BATTERY            |679   |0.172      |
|STREET              |noc      |THEFT              |666   |0.014      |
|STREET              |wieczór  |THEFT              |634   |0.016      |
|APARTMENT           |dzień    |BATTERY            |596   |0.186      |
|STREET              |rano     |THEFT              |568   |0.016      |
|STREET              |rano     |MOTOR VEHICLE THEFT|567   |0.035

+-----------------------------------------------+---------+---------------------------------+------+-----------+----+
|location_description                           |pora_dnia|primary_type                     |liczba|arrest_rate|rank|
+-----------------------------------------------+---------+---------------------------------+------+-----------+----+
|ABANDONED BUILDING                             |dzień    |BURGLARY                         |2     |0.0        |1   |
|ABANDONED BUILDING                             |dzień    |OTHER OFFENSE                    |1     |0.0        |2   |
|ABANDONED BUILDING                             |dzień    |ROBBERY                          |1     |0.0        |3   |
|ABANDONED BUILDING                             |noc      |CRIMINAL DAMAGE                  |1     |0.0        |1   |
|ABANDONED BUILDING                             |noc      |DECEPTIVE PRACTICE               |1     |0.0        |2   |
|ABANDONED BUILDING                             |noc    